# Test Notebook

Smoke test for running Jupyter notebooks against the local `fantasy-player-valuation` environment.

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "analysis":
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(sys.executable)
print(repo_root)

C:\Users\limic\miniconda3\envs\fantasy-player-valuation\python.exe
C:\dev\fantasy_player_valuation


In [2]:
import ffvaluation
import sqlite3

print(ffvaluation.__file__)

C:\dev\fantasy_player_valuation\src\ffvaluation\__init__.py


# USER

In [3]:
db_path = repo_root / "data" / "raw" / "sleeper" / "discovery" / "discovery.sqlite"
print(db_path)

if db_path.exists():
    with sqlite3.connect(db_path) as con:
        counts = {
            table: con.execute(f"select count(*) from {table}").fetchone()[0]
            for table in ["users", "frontier", "leagues", "league_users"]
        }
    counts
else:
    "discovery.sqlite not found"

C:\dev\fantasy_player_valuation\data\raw\sleeper\discovery\discovery.sqlite


In [4]:
def find_sleeper_user(term: str, db_path=db_path):
    """Search discovery users and frontier rows by user id, username, or display name."""
    normalized = term.strip().lower()
    if not normalized:
        raise ValueError("term must not be blank")
    if not db_path.exists():
        raise FileNotFoundError(db_path)

    with sqlite3.connect(db_path) as con:
        con.row_factory = sqlite3.Row
        users = con.execute(
            """
            select user_id, display_name
            from users
            where lower(user_id) = ? or lower(display_name) = ?
            order by display_name, user_id
            """,
            (normalized, normalized),
        ).fetchall()
        frontier = con.execute(
            """
            select user_id, username, display_name, discovered_at,
                   discovered_from_league_id, expanded_at
            from frontier
            where lower(user_id) = ?
               or lower(coalesce(username, '')) = ?
               or lower(coalesce(display_name, '')) = ?
            order by discovered_at, user_id
            """,
            (normalized, normalized, normalized),
        ).fetchall()

    return {
        "users": [dict(row) for row in users],
        "frontier": [dict(row) for row in frontier],
    }


find_sleeper_user("bbroc")

{'users': [{'user_id': '466725566741999616', 'display_name': 'bbroc'}],
 'frontier': [{'user_id': '466725566741999616',
   'username': None,
   'display_name': 'bbroc',
   'discovered_at': '2026-06-12T12:28:32.925154+00:00',
   'discovered_from_league_id': '1335078674310926336',
   'expanded_at': '2026-06-16T12:40:48.173512+00:00'}]}

# TRADES

In [5]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

trade_db_path = repo_root / "data" / "sample" / "sleeper" / "trades" / "sample.sqlite"
print(trade_db_path)

if not trade_db_path.exists():
    raise FileNotFoundError(
        f"{trade_db_path} not found. Run `ffvaluation sample-sleeper-trades` first."
    )

with sqlite3.connect(trade_db_path) as con:
    trades_df = pd.read_sql_query(
        """
        select *
        from trades
        order by created_at desc, league_id, transaction_id
        """,
        con,
    )

print(f"{len(trades_df):,} trades across {trades_df['league_id'].nunique():,} leagues")
trades_df.head()

C:\dev\fantasy_player_valuation\data\sample\sleeper\trades\sample.sqlite
1,651 trades across 92 leagues


,captured_at,league_id,league_name,league_season,previous_league_id,round,transaction_id,status,created,created_at,...,waiver_budget,total_rosters,is_dynasty,is_superflex,ppr,te_premium,target_format_guess,league_settings,scoring_settings,roster_positions
0,2026-06-22T23:27:53.711839+00:00,1179566594926120960,479 Dynasty League,2025,,17,1311431372170076160,complete,1767032113643,2025-12-29T18:15:13.643000+00:00,...,[],12,true,true,1,0,true,"{""bench_lock"":0,""best_ball"":0,""capacity_overri...","{""blk_kick"":2.0,""def_st_ff"":1.0,""def_st_fum_re...","[""QB"",""RB"",""RB"",""WR"",""WR"",""TE"",""FLEX"",""FLEX"",""..."
1,2026-06-22T23:27:53.711839+00:00,1180144557483417600,Transparent League,2025,1048243498023059456,17,1311410045602234368,complete,1767027028994,2025-12-29T16:50:28.994000+00:00,...,[],12,true,true,1,0,true,"{""bench_lock"":0,""best_ball"":0,""capacity_overri...","{""blk_kick"":2.0,""def_st_ff"":1.0,""def_st_fum_re...","[""QB"",""RB"",""RB"",""WR"",""WR"",""TE"",""FLEX"",""FLEX"",""..."
2,2026-06-22T23:27:53.711839+00:00,1179566594926120960,479 Dynasty League,2025,,17,1311080523417800704,complete,1766948464783,2025-12-28T19:01:04.783000+00:00,...,[],12,true,true,1,0,true,"{""bench_lock"":0,""best_ball"":0,""capacity_overri...","{""blk_kick"":2.0,""def_st_ff"":1.0,""def_st_fum_re...","[""QB"",""RB"",""RB"",""WR"",""WR"",""TE"",""FLEX"",""FLEX"",""..."
3,2026-06-22T23:27:53.711839+00:00,1183948306311839744,alt,2025,1048269761618067456,17,1310336290050301952,complete,1766771025721,2025-12-26T17:43:45.721000+00:00,...,[],12,true,true,1,0,true,"{""bench_lock"":0,""best_ball"":0,""capacity_overri...","{""blk_kick"":2.0,""def_st_ff"":1.0,""def_st_fum_re...","[""QB"",""RB"",""RB"",""WR"",""WR"",""TE"",""FLEX"",""SUPER_F..."
4,2026-06-22T23:27:53.711839+00:00,1180144557483417600,Transparent League,2025,1048243498023059456,17,1309974628873015296,complete,1766684798976,2025-12-25T17:46:38.976000+00:00,...,[],12,true,true,1,0,true,"{""bench_lock"":0,""best_ball"":0,""capacity_overri...","{""blk_kick"":2.0,""def_st_ff"":1.0,""def_st_fum_re...","[""QB"",""RB"",""RB"",""WR"",""WR"",""TE"",""FLEX"",""FLEX"",""..."


In [6]:
trades_df.columns

Index(['captured_at', 'league_id', 'league_name', 'league_season',
       'previous_league_id', 'round', 'transaction_id', 'status', 'created',
       'created_at', 'status_updated', 'status_updated_at', 'roster_ids',
       'consenter_ids', 'adds', 'drops', 'draft_picks', 'waiver_budget',
       'total_rosters', 'is_dynasty', 'is_superflex', 'ppr', 'te_premium',
       'target_format_guess', 'league_settings', 'scoring_settings',
       'roster_positions'],
      dtype='str')